# Bulls Analytics - Data Exploration

**Purpose:** Explore what data is available from the NBA API for our Bulls Instagram analytics project.

**Date:** January 2026  
**Phase:** 1 - Foundation

---

## What This Notebook Covers
1. Connecting to the NBA API
2. Finding the Bulls team data
3. Pulling game schedules
4. Getting player box scores (traditional + advanced stats)
5. Understanding what data we have to work with

---


In [29]:
# My first NBA data exploration
from nba_api.stats.static import teams

## Step 1: Setup & Team Lookup

First, we import the NBA API's static data to look up team information.


In [30]:
# Get all NBA teams
all_teams = teams.get_teams()
print(f"Found {len(all_teams)} teams")

Found 30 teams


In [31]:
# What does a team entry look like?
all_teams[0]

{'id': 1610612737,
 'full_name': 'Atlanta Hawks',
 'abbreviation': 'ATL',
 'nickname': 'Hawks',
 'city': 'Atlanta',
 'state': 'Georgia',
 'year_founded': 1949}

In [32]:
# Find the Chicago Bulls
bulls = [team for team in all_teams if team['full_name'] == 'Chicago Bulls'][0]
print(f"Team: {bulls['full_name']}")
print(f"Abbreviation: {bulls['abbreviation']}")
print(f"Team ID: {bulls['id']}")

Team: Chicago Bulls
Abbreviation: CHI
Team ID: 1610612741


In [33]:
# Import the endpoint to get game data
from nba_api.stats.endpoints import leaguegamefinder

# Get only regular season games
# The LeagueGameFinder has a season_type parameter
bulls_regular_season = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=bulls['id'],
    season_nullable='2025-26',
    season_type_nullable='Regular Season'  # Filter to regular season only!
).get_data_frames()[0]

print(f"Regular season games: {len(bulls_regular_season)}")
bulls_games.head()  # Show first 5 games

Regular season games: 36


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612741,CHI,Chicago Bulls,0022500503,2026-01-05,CHI @ BOS,L,240,101,38,86,0.442,16,46,0.348,9,10,0.900,6,39,45,25,2,4,11,13,-14.0
1,22025,1610612741,CHI,Chicago Bulls,0022500489,2026-01-03,CHI vs. CHA,L,240,99,39,87,0.448,11,41,0.268,10,14,0.714,6,37,43,32,9,5,11,16,-13.0
2,22025,1610612741,CHI,Chicago Bulls,0022500480,2026-01-02,CHI vs. ORL,W,241,121,45,90,0.500,14,41,0.341,17,19,0.895,9,38,47,31,6,5,12,15,7.0
3,22025,1610612741,CHI,Chicago Bulls,0022500466,2025-12-31,CHI vs. NOP,W,240,134,47,92,0.511,15,43,0.349,25,34,0.735,8,41,49,33,5,7,7,18,16.0
4,22025,1610612741,CHI,Chicago Bulls,0022500452,2025-12-29,CHI vs. MIN,L,242,101,38,93,0.409,13,38,0.342,12,14,0.857,10,32,42,25,3,7,16,17,-35.0


## Step 2: Get Bulls Game Schedule

Now we fetch all Bulls regular season games using the `LeagueGameFinder` endpoint.


In [34]:
# Get the most recent game ID
# Games are sorted by date, so first row is most recent
latest_game_id = bulls_regular_season.iloc[0]['GAME_ID']
latest_game_date = bulls_regular_season.iloc[0]['GAME_DATE']
latest_matchup = bulls_regular_season.iloc[0]['MATCHUP']

print(f"Most recent game: {latest_matchup}")
print(f"Date: {latest_game_date}")
print(f"Game ID: {latest_game_id}")

Most recent game: CHI @ BOS
Date: 2026-01-05
Game ID: 0022500503


In [35]:
from nba_api.stats.endpoints import boxscoretraditionalv3

# Get detailed stats for that game
box_score = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=latest_game_id)

# The API returns multiple data frames - player stats is the first one
player_stats = box_score.player_stats.get_data_frame()

print(f"Players in this game: {len(player_stats)}")
player_stats.head(10)

Players in this game: 27


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,threePointersMade,threePointersAttempted,threePointersPercentage,freeThrowsMade,freeThrowsAttempted,freeThrowsPercentage,reboundsOffensive,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints
0,0022500503,1610612738,Boston,Celtics,BOS,celtics,1627759,Jaylen,Brown,J. Brown,jaylen-brown,F,,,34:34,6,24,0.250,1,6,0.167,1,2,0.5,3,5,8,4,0,0,3,2,14,12.0
1,0022500503,1610612738,Boston,Celtics,BOS,celtics,1630573,Sam,Hauser,S. Hauser,sam-hauser,F,,,20:36,3,8,0.375,3,7,0.429,0,0,0.0,0,1,1,1,1,0,0,2,9,6.0
2,0022500503,1610612738,Boston,Celtics,BOS,celtics,1629674,Neemias,Queta,N. Queta,neemias-queta,C,,,24:35,6,9,0.667,0,0,0.000,1,1,1.0,6,7,13,1,1,1,1,1,13,11.0
3,0022500503,1610612738,Boston,Celtics,BOS,celtics,1630202,Payton,Pritchard,P. Pritchard,payton-pritchard,G,,,31:15,8,17,0.471,4,7,0.571,1,1,1.0,0,6,6,5,1,0,0,1,21,11.0
4,0022500503,1610612738,Boston,Celtics,BOS,celtics,1628401,Derrick,White,D. White,derrick-white,G,,,37:03,3,15,0.200,1,10,0.100,3,3,1.0,1,5,6,7,3,1,0,1,10,0.0
5,0022500503,1610612738,Boston,Celtics,BOS,celtics,1630568,Luka,Garza,L. Garza,luka-garza,,,,23:25,3,4,0.750,0,0,0.000,2,2,1.0,3,1,4,0,1,2,0,0,8,3.0
6,0022500503,1610612738,Boston,Celtics,BOS,celtics,1642864,Hugo,González,H. González,hugo-gonzález,,,,12:19,1,5,0.200,1,4,0.250,2,2,1.0,4,3,7,1,0,0,0,1,5,5.0
7,0022500503,1610612738,Boston,Celtics,BOS,celtics,1629014,Anfernee,Simons,A. Simons,anfernee-simons,,,,27:10,9,16,0.563,8,14,0.571,1,2,0.5,2,1,3,3,0,0,3,0,27,10.0
8,0022500503,1610612738,Boston,Celtics,BOS,celtics,1641775,Jordan,Walsh,J. Walsh,jordan-walsh,,,,18:24,2,4,0.500,1,2,0.500,0,0,0.0,1,4,5,0,0,0,0,1,5,7.0
9,0022500503,1610612738,Boston,Celtics,BOS,celtics,1631248,Baylor,Scheierman,B. Scheierman,baylor-scheierman,,,,10:39,1,2,0.500,1,2,0.500,0,0,0.0,0,1,1,0,0,0,0,2,3,5.0


In [37]:
# Filter to Bulls players only
# V3 uses 'teamId' (camelCase) instead of 'TEAM_ID'
bulls_players = player_stats[player_stats['teamId'] == bulls['id']]

# Create a full name column and select key stats
# V3 column names are different!
bulls_players = bulls_players.copy()
bulls_players['playerName'] = bulls_players['firstName'] + ' ' + bulls_players['familyName']

key_stats = bulls_players[['playerName', 'minutes', 'points', 'reboundsTotal', 'assists', 
                            'steals', 'blocks', 'fieldGoalsMade', 'fieldGoalsAttempted',
                            'threePointersMade', 'threePointersAttempted']]
key_stats.sort_values('points', ascending=False)

,playerName,minutes,points,reboundsTotal,assists,steals,blocks,fieldGoalsMade,fieldGoalsAttempted,threePointersMade,threePointersAttempted
16,Matas Buzelis,30:58,26,3,2,0,1,9,12,3,4
17,Nikola Vučević,36:21,15,15,7,1,0,6,14,3,5
21,Ayo Dosunmu,27:48,15,3,5,0,1,6,13,3,7
19,Tre Jones,23:22,10,5,4,0,0,4,7,0,1
18,Kevin Huerter,19:39,8,3,0,0,1,4,9,0,4
25,Jevon Carter,11:46,6,1,1,0,0,2,6,2,5
24,Julian Phillips,13:14,6,2,0,1,0,2,5,2,4
15,Isaac Okoro,24:52,5,3,1,0,1,1,7,1,7
20,Coby White,25:23,5,6,3,0,0,2,7,1,5
26,Dalen Terry,4:55,3,0,0,0,0,1,1,1,1


## Step 3: Get Player Box Score Data

For any specific game, we can get detailed player stats using the box score endpoints.

**Key Learning:** The NBA API deprecated V2 endpoints for the 2025-26 season. Use V3 endpoints instead (e.g., `boxscoretraditionalv3`).


In [38]:
# Let's see everything we already have
print("All stats from basic box score:")
for col in player_stats.columns:
    print(f"  - {col}")

All stats from basic box score:
  - gameId
  - teamId
  - teamCity
  - teamName
  - teamTricode
  - teamSlug
  - personId
  - firstName
  - familyName
  - nameI
  - playerSlug
  - position
  - comment
  - jerseyNum
  - minutes
  - fieldGoalsMade
  - fieldGoalsAttempted
  - fieldGoalsPercentage
  - threePointersMade
  - threePointersAttempted
  - threePointersPercentage
  - freeThrowsMade
  - freeThrowsAttempted
  - freeThrowsPercentage
  - reboundsOffensive
  - reboundsDefensive
  - reboundsTotal
  - assists
  - steals
  - blocks
  - turnovers
  - foulsPersonal
  - points
  - plusMinusPoints


In [39]:
from nba_api.stats.endpoints import boxscoreadvancedv3, boxscoremiscv3, boxscorescoringv3

# Advanced stats (offensive rating, usage rate, etc.)
advanced = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=latest_game_id)
advanced_stats = advanced.player_stats.get_data_frame()
print("ADVANCED STATS columns:")
print(advanced_stats.columns.tolist())

ADVANCED STATS columns:
['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']


In [40]:
# Miscellaneous stats
misc = boxscoremiscv3.BoxScoreMiscV3(game_id=latest_game_id)
misc_stats = misc.player_stats.get_data_frame()
print("\nMISC STATS columns:")
print(misc_stats.columns.tolist())


MISC STATS columns:
['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'pointsOffTurnovers', 'pointsSecondChance', 'pointsFastBreak', 'pointsPaint', 'oppPointsOffTurnovers', 'oppPointsSecondChance', 'oppPointsFastBreak', 'oppPointsPaint', 'blocks', 'blocksAgainst', 'foulsPersonal', 'foulsDrawn']


In [41]:
# Scoring breakdown
scoring = boxscorescoringv3.BoxScoreScoringV3(game_id=latest_game_id)
scoring_stats = scoring.player_stats.get_data_frame()
print("\nSCORING BREAKDOWN columns:")
print(scoring_stats.columns.tolist())


SCORING BREAKDOWN columns:
['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'percentageFieldGoalsAttempted2pt', 'percentageFieldGoalsAttempted3pt', 'percentagePoints2pt', 'percentagePointsMidrange2pt', 'percentagePoints3pt', 'percentagePointsFastBreak', 'percentagePointsFreeThrow', 'percentagePointsOffTurnovers', 'percentagePointsPaint', 'percentageAssisted2pt', 'percentageUnassisted2pt', 'percentageAssisted3pt', 'percentageUnassisted3pt', 'percentageAssistedFGM', 'percentageUnassistedFGM']


## Step 4: Explore Available Stats

The NBA API has multiple box score endpoints, each with different stats:

| Endpoint | What It Contains |
|----------|------------------|
| `boxscoretraditionalv3` | Basic stats (PTS, REB, AST, etc.) |
| `boxscoreadvancedv3` | Advanced metrics (Usage %, ORtg, DRtg, TS%) |
| `boxscoremiscv3` | Misc stats (Points in paint, fast break, etc.) |
| `boxscorescoringv3` | Scoring breakdown (% assisted, shot distribution) |


In [42]:
import pandas as pd

# Make pandas display all columns nicely
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Merge the interesting stats together (they all have 'personId' in common)
bulls_traditional = player_stats[player_stats['teamId'] == bulls['id']].copy()
bulls_advanced = advanced_stats[advanced_stats['teamId'] == bulls['id']].copy()
bulls_misc = misc_stats[misc_stats['teamId'] == bulls['id']].copy()
bulls_scoring = scoring_stats[scoring_stats['teamId'] == bulls['id']].copy()

# Create player name
bulls_traditional['player'] = bulls_traditional['firstName'] + ' ' + bulls_traditional['familyName']

# Select the most interesting columns from each
combined = bulls_traditional[['player', 'minutes', 'points', 'reboundsTotal', 'assists', 
                               'steals', 'blocks', 'turnovers', 'plusMinusPoints',
                               'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
                               'threePointersMade', 'threePointersAttempted']].copy()

# Add some advanced stats
combined['usageRate'] = bulls_advanced['usagePercentage'].values
combined['trueShootingPct'] = bulls_advanced['trueShootingPercentage'].values
combined['offRating'] = bulls_advanced['offensiveRating'].values
combined['defRating'] = bulls_advanced['defensiveRating'].values

# Add scoring breakdown
combined['pctPointsPaint'] = bulls_scoring['percentagePointsPaint'].values
combined['pctAssisted'] = bulls_scoring['percentageAssistedFGM'].values

# Sort by points and display
combined = combined.sort_values('points', ascending=False)
print(f"📊 BULLS vs CELTICS - {latest_game_date}")
print("=" * 80)
combined

📊 BULLS vs CELTICS - 2026-01-05


,player,minutes,points,reboundsTotal,assists,steals,blocks,turnovers,plusMinusPoints,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,threePointersMade,threePointersAttempted,usageRate,trueShootingPct,offRating,defRating,pctPointsPaint,pctAssisted
16,Matas Buzelis,30:58,26,3,2,0,1,2,-6.0,9,12,0.750,3,4,0.229,0.888,112.9,126.7,0.462,0.667
17,Nikola Vučević,36:21,15,15,7,1,0,2,-10.0,6,14,0.429,3,5,0.200,0.536,109.6,120.0,0.267,0.833
21,Ayo Dosunmu,27:48,15,3,5,0,1,0,-4.0,6,13,0.462,3,7,0.210,0.577,120.0,127.3,0.400,0.667
19,Tre Jones,23:22,10,5,4,0,0,1,-7.0,4,7,0.571,0,1,0.180,0.635,100.0,110.4,0.600,0.000
18,Kevin Huerter,19:39,8,3,0,0,1,1,-12.0,4,9,0.444,0,4,0.238,0.444,82.1,112.8,0.750,0.750
25,Jevon Carter,11:46,6,1,1,0,0,0,-1.0,2,6,0.333,2,5,0.250,0.500,117.4,121.7,0.000,1.000
24,Julian Phillips,13:14,6,2,0,1,0,0,-9.0,2,5,0.400,2,4,0.167,0.600,85.7,117.9,0.000,1.000
15,Isaac Okoro,24:52,5,3,1,0,1,1,-6.0,1,7,0.143,1,7,0.158,0.317,107.8,117.3,0.000,1.000
20,Coby White,25:23,5,6,3,0,0,4,-5.0,2,7,0.286,1,5,0.196,0.357,120.0,130.0,0.400,0.500
26,Dalen Terry,4:55,3,0,0,0,0,0,-1.0,1,1,1.000,1,1,0.091,1.500,120.0,130.0,0.000,1.000
